## Markdown Scrape & Export (Files Only)

Fetch markdown files from GitHub (`uvarc/rc-learning`) and save them as local `.md` files.

This trims `scrape-md-new.ipynb` down to only the fetch + export steps, dropping the DB/vector-store ingestion — matching the pattern used in `scrape-jira-new.ipynb`'s markdown-only variant.

## 1. Fetch Markdown from GitHub

Uses the GitHub tree API to list all `.md` files under `content/`, then fetches each via raw URL.
Skips drafts (`draft = true` / `draft: true` in frontmatter).

In [1]:
import requests

GITHUB_REPO = "uvarc/rc-learning"
GITHUB_BRANCH = "main"
CONTENT_PATH = "content"


def load_documents_from_github(
    repo=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    content_path=CONTENT_PATH
):
    tree_url = (
        f"https://api.github.com/repos/{repo}/git/trees/"
        f"{branch}?recursive=1"
    )

    resp = requests.get(tree_url, timeout=30)
    resp.raise_for_status()
    tree = resp.json().get("tree", [])

    md_files = [
        item for item in tree
        if item["type"] == "blob"
        and item["path"].startswith(content_path + "/")
        and item["path"].endswith(".md")
    ]

    print(f"Found {len(md_files)} markdown files in {repo}/{content_path}")

    documents = []

    for item in md_files:
        raw_url = (
            f"https://raw.githubusercontent.com/"
            f"{repo}/{branch}/{item['path']}"
        )

        r = requests.get(raw_url, timeout=15)

        if r.status_code != 200:
            print(f"Failed to fetch {item['path']}: {r.status_code}")
            continue

        text = r.text

        # Skip TOML-frontmatter drafts
        if text.startswith("+++"):
            end = text.find("+++", 3)
            if end != -1 and "draft = true" in text[3:end]:
                print(f"Skipping draft: {item['path']}")
                continue

        # Skip YAML-frontmatter drafts
        if text.startswith("---"):
            end = text.find("---", 3)
            if end != -1 and "draft: true" in text[3:end]:
                print(f"Skipping draft: {item['path']}")
                continue

        documents.append({
            "source": item["path"],
            "content": text
        })

    # Deduplicate by content
    unique = {
        doc["content"]: doc
        for doc in documents
    }

    documents = list(unique.values())

    print(f"Loaded {len(documents)} documents (after dedup)")

    return documents


markdown_documents = load_documents_from_github()

Found 707 markdown files in uvarc/rc-learning/content


Loaded 706 documents (after dedup)


## 2. Preview

In [2]:
for doc in markdown_documents[:3]:
    print(f"=== {doc['source']} ===")
    print(doc["content"][:300])
    print()

=== content/authors/abd/_index.md ===
---
# Display name
title: Angela Boakye Danquah

# Username (this should match the folder name)
authors:
- abd

# Is this the primary user of the site?
superuser: false

user_groups: 
- DAC

# Role/position
role: Research Computing Scientist

# Organizations/Affiliations
organizations:
- name: Unive

=== content/authors/as/_index.md ===
---
# Display name
title: Ahmad Sheikhzada

# Username (this should match the folder name)
authors:
- as

# Is this the primary user of the site?
superuser: false

# Role/position
role: Technical Support Manager

user_groups:
- RC

# Organizations/Affiliations
organizations:
- name: University of Vi

=== content/authors/bmr/_index.md ===
---
# Display name
title: Bruce Rushing

# Username (this should match the folder name)
authors:
- bmr

# Is this the primary user of the site?
superuser: false

user_groups: 
- DAC

# Role/position
role: Research Computing Scientist

# Organizations/Affiliations
organizations:
- na

## 3. Generate Markdown Files

In [3]:
from pathlib import Path
import re

# Locate this repo from either the project root or a notebook subfolder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "app" / "kb_integration" / "tasks.py").is_file()
     and (path / "scrapers").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or a subfolder.")

# Store generated files inside this repo
OUTPUT_FOLDER = PROJECT_ROOT / "data" / "markdown"

# Automatically create data/markdown if it doesn't exist
OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

def safe_filename(filename):

    filename = re.sub(
        r'[<>:"\\|?*]',
        '_',
        filename
    )

    filename = re.sub(
        r'\s+',
        ' ',
        filename
    ).strip()

    return filename[:150]


print(
    f"Documents available: "
    f"{len(markdown_documents)}"
)

print(
    f"Writing files to: "
    f"{OUTPUT_FOLDER}"
)


created = 0
failed = 0


for doc in markdown_documents:

    try:
        relative_path = doc["source"].split("content/", 1)[-1]

        flat_name = safe_filename(
            relative_path.removesuffix(".md").replace("/", "_")
        )

        file_path = OUTPUT_FOLDER / f"{flat_name}.md"

        with open(
            file_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                doc["content"].strip() + "\n"
            )

        created += 1

        print(
            f"Created: {file_path.name}"
        )

    except Exception as e:

        failed += 1

        print(
            f"ERROR processing "
            f"{doc.get('source', 'unknown')}: "
            f"{e}"
        )


print("Markdown Generation Complete")

print(f"Created: {created} files")
print(f"Failed:  {failed} files")
print(f"Output folder: {OUTPUT_FOLDER}")

Documents available: 706
Writing files to: C:\Users\mayoe\OneDrive\Desktop\kb_Upload\rag-kb-upload\data\markdown
Created: authors_abd__index.md
Created: authors_as__index.md
Created: authors_bmr__index.md
Created: authors_cag__index.md
Created: authors_cmd__index.md
Created: authors_dat__index.md
Created: authors_gka__index.md
Created: authors_hp__index.md
Created: authors_jmh__index.md
Created: authors_kah__index.md
Created: authors_kal__index.md
Created: authors_khs__index.md
Created: authors_mab__index.md
Created: authors_pbo__index.md
Created: authors_ppr__index.md
Created: authors_rs__index.md
Created: authors_teh__index.md
Created: authors_uvarc__index.md
Created: authors_wtr__index.md
Created: courses__index.md
Created: courses_cpp-introduction__index.md
Created: courses_cpp-introduction_advanced_io.md
Created: courses_cpp-introduction_boost.md
Created: courses_cpp-introduction_building.md
Created: courses_cpp-introduction_c_arrays.md
Created: courses_cpp-introduction_characters

Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_29.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_35.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_39.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_4.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_41.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_45.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_48.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_52.md
Created: notes_bioinfo-reproducibility_bioinfo-reproducibility_9.md
Created: notes_bioinfo-reproducibility_src_out.md
Created: notes_bioinfo-tools-riv_index.md
Created: notes_building-running-c-cpp-fortran__index.md
Created: notes_building-running-c-cpp-fortran_build_multiple_files.md
Created: notes_building-running-c-cpp-fortran_build_single_file.md
Created: notes_building-running-c-cpp-fortran_cmake.md
Created: notes_building-running-c-cpp-for

Created: notes_dl-drug-discovery_featurization.md
Created: notes_dl-drug-discovery_regression-example.md
Created: notes_dl-drug-discovery_smile-format.md
Created: notes_dl-drug-discovery_summary.md
Created: notes_dl-drug-discovery_therapeutics.md
Created: notes_genomics_1-history.md
Created: notes_genomics_2-construction.md
Created: notes_genomics_3-technologies.md
Created: notes_genomics_4-formats.md
Created: notes_genomics_5-genomes.md
Created: notes_genomics_6-modules.md
Created: notes_genomics_7-0-demos.md
Created: notes_genomics_7-1-busco.md
Created: notes_genomics_7-2-repeatmasker.md
Created: notes_genomics_7-3-stringtie.md
Created: notes_genomics_7-4-smrtlink.md
Created: notes_genomics_8-software.md
Created: notes_genomics_9-more.md
Created: notes_genomics__index.md
Created: notes_genomics_tutorial_landing_index.md
Created: notes_git-intro_index.md
Created: notes_globus-data-transfer__index.md
Created: notes_globus-data-transfer_add_locations.md
Created: notes_globus-data-transf

Created: notes_seurat-bioinformatics_03-seurat-intro.md
Created: notes_seurat-bioinformatics_04-rstudio-portal.md
Created: notes_seurat-bioinformatics_05-setup.md
Created: notes_seurat-bioinformatics_06-dataloading.md
Created: notes_seurat-bioinformatics_07-qc.md
Created: notes_seurat-bioinformatics_08-filter-normalize.md
Created: notes_seurat-bioinformatics_09-variable-features.md
Created: notes_seurat-bioinformatics_10-scale.md
Created: notes_seurat-bioinformatics_11-pca.md
Created: notes_seurat-bioinformatics_12-visualize-pca.md
Created: notes_seurat-bioinformatics_13-dimensionality.md
Created: notes_seurat-bioinformatics_14-clustering-analys.md
Created: notes_seurat-bioinformatics_15-umap.md
Created: notes_seurat-bioinformatics_16-biomarkers.md
Created: notes_seurat-bioinformatics_17-expression-vis.md
Created: notes_seurat-bioinformatics_18-biomarkers.md
Created: notes_seurat-bioinformatics_19-modifyingplots.md
Created: notes_seurat-bioinformatics_20-recap.md
Created: notes_seurat-

Created: tutorials_hpc-intro_index.md
Created: tutorials_interactive-apps-ood_index.md
Created: tutorials_pytorch-hpc_index.md
Created: tutorials_rio-intro_index.md
Created: tutorials_slurm-from-cli_index.md
Created: tutorials_unix-tutorial_index.md
Created: tutorials_using-the-shell_index.md
Created: tutorials_uva-rc-genai_index.md
Created: tutorials_vscode-intro_index.md
Created: tutorials_working-with-files_index.md
Created: tutorials_working-with-jobs-ood_index.md
Created: videos_example_index.md
Created: videos_matlab-data-science_index.md
Created: videos_matlab-parallel_index.md
Markdown Generation Complete
Created: 706 files
Failed:  0 files
Output folder: C:\Users\mayoe\OneDrive\Desktop\kb_Upload\rag-kb-upload\data\markdown
